# Reading Wikipedia with `esperantilo`

This notebook fetches one Wikipedia article and shows what the
[`esperantilo`](https://github.com/jparisu/nlp-esperantilo) library gives you back.

You will be asked for two things:

| Input | What it is | Example |
| --- | --- | --- |
| **Title** | The title of the page, as it appears on Wikipedia | `Esperanto` |
| **Language** | The two-letter code of the Wikipedia to read | `eo`, `es`, `en` |

Run the cells in order, from top to bottom (`Shift`+`Enter`).

## 1. Install the library

The library is not on PyPI, so `pip` installs it straight from GitHub.
This takes a few seconds and only has to be done once per session.

In [ ]:
LIBRARY = "esperantilo"
REPO = "https://github.com/jparisu/nlp-esperantilo.git"

# Which version to install. "main" is the released code; put a branch or a tag
# name here to try something else.
BRANCH = "jparisu/v0.1"

# INSTALL LIBRARY
import subprocess
import sys

# Check if installed
if LIBRARY in sys.modules:
    print(f"{LIBRARY} already installed.")
else:
    try:
        subprocess.check_call(["pip", "install", f"{REPO}@{SIARENA_BRANCH}"])
        print(f"{LIBRARY} Package installed.")
    except Exception as e:
        print(f"Error installing the package: {e}. Installing by magic command.")
        %pip install --upgrade git+{REPO}@{BRANCH}

## 2. Choose the page

Edit the two constants below and run the cell. Everything after it uses
whatever you set here, so re-run this cell and the ones under it to look up a
different page.

In [ ]:
# Set the title and language for the Wikipedia search.
TITLE = "Esperanto"

# Set the language code for the Wikipedia search. This should be a two-letter:
# - es: Spanish
# - eo: Esperanto
# - en: English
LANGUAGE = "eo"

print(f"\nLooking for {TITLE!r} in the {LANGUAGE!r} Wikipedia...")

## 3. Fetch the article

`WikiPage.look_up` does the whole job: it finds the page, downloads it and
returns it as a `WikiPage` object, with the wiki markup already stripped.

If the language you asked for does not have that title, it tries Spanish,
English and Esperanto in turn, so you usually get *something* back. Note the
language of the result below — it may not be the one you typed.

In [ ]:
from esperantilo import WikiPage

page = WikiPage.look_up(TITLE, language=LANGUAGE)

# A WikiPage prints itself as a one-line summary.
page

## 4. What came back

Five things: the title Wikipedia actually served (redirects are followed, so it
may differ from what you typed), the language, the **QID** — the identifier
Wikidata gives to the *concept*, the same one in every language — the address of
the article, and how many sections it has.

In [ ]:
print("Title:   ", page.title)
print("Language:", page.language)
print("QID:     ", page.qid)
print("URL:     ", page.url)

# len(page) is the number of sections, not the length of the text.
print("Sections:", len(page))

## 5. The sections

`section_names` lists the headings, in the order the article presents them. The
first one is the *lead* — the text before any heading — which Wikipedia leaves
untitled, so it is stored under the title of the page.

In [ ]:
for number, name in enumerate(page.section_names, start=1):
    # len(page.section(name)) is how many characters that section holds.
    print(f"{number:3}. {name}  ({len(page.section(name))} characters)")

## 6. Read one section

`page.section(name)` returns the text of a single section. The name is matched
ignoring case, so `"historio"` finds `"Historio"`. `page[name]` does the same
thing, and `name in page` tells you whether a section exists.

In [ ]:
# Change this to any heading listed above, in any capitalisation.
wanted = page.section_names[0]

if wanted in page:
    print(f"== {wanted} ==\n")
    print(page.section(wanted))
else:
    print(f"This article has no {wanted!r} section.")

## 7. The whole article

`full_text()` joins every section back together, headings included. This is the
plain text you would feed to a tokenizer.

Articles get long, so only the beginning is printed here.

In [ ]:
text = page.full_text()

print(f"{len(text)} characters in total. First 2000:\n")
print(text[:2000])

## 8. Next step: sentence segmentation

Fetching the text is half the job. The other half of the library,
`sentence_tokenizer`, splits it into sentences — the first step of almost
every NLP pipeline.

A sentence ends at `.`, `!` or `?`, and the character stays with the sentence
it closes.

In [ ]:
from esperantilo import sentence_tokenizer

# The lead of the article: the text before the first heading, which is stored
# under the title of the page.
lead = page.section(page.title)

sentences = sentence_tokenizer(lead)

print(f"{len(sentences)} sentences in the lead. First 5:\n")
for number, sentence in enumerate(sentences[:5], start=1):
    print(f"{number}. {sentence}\n")

The tokenizer is deliberately naive: every terminator cuts, whatever surrounds
it. Look through the sentences above and you will probably find one split in
the wrong place — an abbreviation, a decimal number or a domain name.

That is a known limitation, not a mystery bug. Fixing it is the kind of
exercise the course prepares you for.

## What to try next

- Run step 2 again with another title or language — everything below it updates.
- Ask for a title that does not exist in your language, such as `Perejil` with
  `eo`, and watch which language answers instead: `page.language` tells you.
- Tokenize a different section, or the whole article with `page.full_text()`.

Wikipedia text is **CC BY-SA**: if you republish it, keep the attribution and
the licence. `page.url` is the link to credit.